In [2]:
ml bcftools/1.20.0
ml vcftools/0.1.16 
ml samtools/1.21

In [3]:
cd ../../../../hybrid_diversity/5_snp_analyses/4_chloroplast

In [3]:
#bcftools stats chloro_variants_raw.vcf | grep "^SN"
#bcftools query -l chloro_variants.renamed.vcf | wc -l
#bcftools query -l chloro_variants_raw.vcf | sed 's|.*/04_markduplicates/||' | sed 's/_markduplicates\.bam//' > strain_ids.txt 

### Quality filter to remove low quality and heterozygous sites
402 SNPs pass qual filtering at minQ 30 - 367 of those are homozygous
    
401 SNPs pass qual filtering at minQ 100 - 367 of those are homozygous

In [4]:
bcftools reheader -s strain_ids.txt -o chloro_variants.renamed.vcf chloro_variants_raw.vcf

In [5]:
vcftools --vcf chloro_variants.renamed.vcf \
  --remove-indels --min-alleles 2 --max-alleles 2 \
  --minQ 30 \
  --min-meanDP 20 \
  --max-missing 0.9 \
  --recode --out chloro_variants_qual30


VCFtools - 0.1.16
(C) Adam Auton and Anthony Marcketta 2009

Parameters as interpreted:
	--vcf chloro_variants.renamed.vcf
	--max-alleles 2
	--min-alleles 2
	--min-meanDP 20
	--minQ 30
	--max-missing 0.9
	--out chloro_variants_qual30
	--recode
	--remove-indels

After filtering, kept 32 out of 32 Individuals
Outputting VCF file...
After filtering, kept 402 out of a possible 65808 Sites
Run Time = 1.00 seconds


In [5]:
vcftools --vcf chloro_variants.renamed.vcf \
  --remove-indels --min-alleles 2 --max-alleles 2 \
  --minQ 30 \
  --min-meanDP 20 \
  --max-missing 1 \
  --recode --out chloro_variants_qual30_nomissing


VCFtools - 0.1.16
(C) Adam Auton and Anthony Marcketta 2009

Parameters as interpreted:
	--vcf chloro_variants.renamed.vcf
	--max-alleles 2
	--min-alleles 2
	--min-meanDP 20
	--minQ 30
	--max-missing 1
	--out chloro_variants_qual30_nomissing
	--recode
	--remove-indels

After filtering, kept 32 out of 32 Individuals
Outputting VCF file...
After filtering, kept 239 out of a possible 65808 Sites
Run Time = 0.00 seconds


In [14]:
vcftools --vcf chloro_variants.renamed.vcf \
  --remove-indels --min-alleles 2 --max-alleles 2 \
  --minQ 30 \
  --min-meanDP 20 \
  --max-missing 1 \
  --maf 0.05 \
  --recode --out chloro_variants_qual30_maf5_nomissing


VCFtools - 0.1.16
(C) Adam Auton and Anthony Marcketta 2009

Parameters as interpreted:
	--vcf chloro_variants.renamed.vcf
	--maf 0.05
	--max-alleles 2
	--min-alleles 2
	--min-meanDP 20
	--minQ 30
	--max-missing 1
	--out chloro_variants_qual30_maf5_nomissing
	--recode
	--remove-indels

After filtering, kept 32 out of 32 Individuals
Outputting VCF file...
After filtering, kept 211 out of a possible 65808 Sites
Run Time = 1.00 seconds


In [10]:
# exclude any site with at least one het call
bcftools filter -e 'GT[*]="het"' chloro_variants_qual30.recode.vcf \
    -o chloro_variants_qual30_nohets.vcf

In [10]:
# exclude any site with at least one het call
bcftools filter -e 'GT[*]="het"' chloro_variants_qual30_nomissing.recode.vcf \
    -o chloro_variants_qual30_nomissing_nohets.vcf

In [15]:
# exclude any site with at least one het call
bcftools filter -e 'GT[*]="het"' chloro_variants_qual30_maf5_nomissing.recode.vcf \
    -o chloro_variants_qual30_maf5_nomissing_nohets.vcf

In [16]:
bcftools stats chloro_variants_qual30_maf5_nomissing_nohets.vcf | grep "^SN"

SN	0	number of samples:	32
SN	0	number of records:	188
SN	0	number of no-ALTs:	0
SN	0	number of SNPs:	188
SN	0	number of MNPs:	0
SN	0	number of indels:	0
SN	0	number of others:	0
SN	0	number of multiallelic sites:	0
SN	0	number of multiallelic SNP sites:	0


In [17]:
# chloroplast contig length = 107,829bp
vcftools --vcf chloro_variants_qual30_nohets.vcf --site-pi --out chloro_variants_qual30nohet_allstrains 2>/dev/null
awk '{sum+=$3} END {print "Mean π " pop ":", sum/107829}' chloro_variants_qual30nohet_allstrains.sites.pi

awk: /opt/apps/anaconda3/22.10.0/lib/./libtinfo.so.6: no version information available (required by /opt/apps/anaconda3/22.10.0/lib/libreadline.so.8)
Mean π : 0.00131904


In [18]:
vcftools --vcf chloro_variants_qual30_nohets.vcf --keep strain_ids_A1_chloroplasts.txt --site-pi --out chloro_variants_qual30nohet_A1strains 2>/dev/null
awk '{sum+=$3} END {print "Mean π " pop ":", sum/107829}' chloro_variants_qual30nohet_A1strains.sites.pi

awk: /opt/apps/anaconda3/22.10.0/lib/./libtinfo.so.6: no version information available (required by /opt/apps/anaconda3/22.10.0/lib/libreadline.so.8)
Mean π : 0.000124724


In [19]:
vcftools --vcf chloro_variants_qual30_nohets.vcf --keep strain_ids_A2_chloroplasts.txt --site-pi --out chloro_variants_qual30nohet_A2strains 2>/dev/null
awk '{sum+=$3} END {print "Mean π " pop ":", sum/107829}' chloro_variants_qual30nohet_A2strains.sites.pi

awk: /opt/apps/anaconda3/22.10.0/lib/./libtinfo.so.6: no version information available (required by /opt/apps/anaconda3/22.10.0/lib/libreadline.so.8)
Mean π : 0.000119077


In [17]:
# filter vcf to only include hybrids
vcftools --vcf chloro_variants_qual30_maf5_nomissing_nohets.vcf \
  --remove-indv CCMP2941 \
  --remove-indv RCC3703 \
  --remove-indv CC2192B4 \
  --remove-indv CCMP3037 \
  --remove-indv PK2298B4 \
  --remove-indv CC2192B5 \
  --remove-indv CC2192C1 \
  --remove-indv CC2192D3 \
  --remove-indv CC2192D2 \
  --remove-indv 12B1 \
  --recode --out chloro_variants_qual30_maf5_nomissing_nohets_justhybrids


VCFtools - 0.1.16
(C) Adam Auton and Anthony Marcketta 2009

Parameters as interpreted:
	--vcf chloro_variants_qual30_maf5_nomissing_nohets.vcf
	--out chloro_variants_qual30_maf5_nomissing_nohets_justhybrids
	--recode
	--remove-indv 12B1
	--remove-indv CC2192B4
	--remove-indv CC2192B5
	--remove-indv CC2192C1
	--remove-indv CC2192D2
	--remove-indv CC2192D3
	--remove-indv CCMP2941
	--remove-indv CCMP3037
	--remove-indv PK2298B4
	--remove-indv RCC3703

Excluding individuals in 'exclude' list
After filtering, kept 22 out of 32 Individuals
Outputting VCF file...
After filtering, kept 188 out of a possible 188 Sites
Run Time = 0.00 seconds


In [18]:
ls -ltrh

total 82M
-rw-r----- 1 jennifer.wisecaver its_p_sys_ur_kam-wisecaver  37K Feb 22 13:01 chloro_variants_filtered_renamed.vcf.gz
-rw-r----- 1 jennifer.wisecaver its_p_sys_ur_kam-wisecaver  39M Feb 22 13:01 chloro_variants_raw.vcf
-rw-r----- 1 jennifer.wisecaver its_p_sys_ur_kam-wisecaver  179 Feb 22 23:29 chloro_variants_filtered_renamed.vcf.gz.csi
drwxr-s--- 1 jennifer.wisecaver its_p_sys_ur_kam-wisecaver    0 Mar 31 20:54 archive
-rw-r----- 1 jennifer.wisecaver its_p_sys_ur_kam-wisecaver  268 Mar 31 21:00 strain_ids.txt
-rw-r----- 1 jennifer.wisecaver its_p_sys_ur_kam-wisecaver  13K Mar 31 22:12 chloro_variants_qual30_allstrains.sites.pi
-rw-r----- 1 jennifer.wisecaver its_p_sys_ur_kam-wisecaver  182 Mar 31 22:22 strain_ids_A1_chloroplasts.txt
-rw-r----- 1 jennifer.wisecaver its_p_sys_ur_kam-wisecaver   86 Mar 31 22:23 strain_ids_A2_chloroplasts.txt
-rw-r----- 1 jennifer.wisecaver its_p_sys_ur_kam-wisecaver  11K Mar 31 22:24 chloro_variants_qual30_A1strains.sites.pi
-rw-r----- 1 jennif